# `lic_dsf.pv` — PV_Base instruments

`internal()` / `external()` return **pandas DataFrames** shaped like `PV_Base`.
Load Input 4 with `load_instruments_from_workbook`, or build instruments by hand.

See `docs/03-pv-instruments.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

from lic_dsf.pv import PresentValueInstrument

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. One instrument → internal / external DataFrames (works)

- `internal()` — unit loan of 100; `Term` column holds grace/maturity/rates; year columns hold the schedule
- `external()` — Output metrics only; years are column headers (not a data row)

In [2]:
years = tuple(range(2024, 2024 + 12))
eurobond = PresentValueInstrument(
    name="Eurobond",
    grace=9,
    maturity=12,
    interest_rate=0.09,
    discount_rate=0.05,
    disbursements=[0.0, 0.0, 0.0, 250.0, 250.0, 250.0, 1000.0, 333.3, 333.3, 333.3, 0.0, 333.3],
    years=years,
    horizon=12,
)

internal = eurobond.internal()   # pd.DataFrame
external = eurobond.external()   # pd.DataFrame

display(internal.iloc[:, :8])
display(external.iloc[:, :8])

print("unit PV[2024]=", float(internal.loc["PV of debt", 2024]))
print("output PV[2024]=", float(external.loc[f"PV of debt   {eurobond.name}", 2024]))

,Term,2024,2025,2026,2027,2028,2029,2030
Eurobond,<NA>,0,1,2,3,4,5,6
Grace Eurobond,9,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Maturity Eurobond / Base,12,100.0000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Interest Eurobond / Debt stock,0.0900,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000
Discount Eurobond / Amortization,0.0500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Interest,<NA>,0.0000,9.0000,9.0000,9.0000,9.0000,9.0000,9.0000
Total debt service,<NA>,0.0000,9.0000,9.0000,9.0000,9.0000,9.0000,9.0000
PV of debt,<NA>,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000
Grant element,<NA>,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
t-g>0,<NA>,0,0,0,0,0,0,0


,2024,2025,2026,2027,2028,2029,2030,2031
"New forex borrowing (gross, USD)",0.0000,0.0000,0.0000,250.0000,250.0000,250.0000,"1,000.0000",333.3000
cumulative,0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3000"
Stock of new forex debt (in USD),0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3000"
PV of debt Eurobond,0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3000"
Total debt service (in USD),0.0000,0.0000,0.0000,0.0000,22.5000,45.0000,67.5000,157.5000
Interest,0.0000,0.0000,0.0000,0.0000,22.5000,45.0000,67.5000,157.5000
Amortization,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


unit PV[2024]= 100.0
output PV[2024]= 0.0


## 2. Load instruments from workbook (target)

```python
instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=False,
)
# each item is a PresentValueInstrument; .internal() / .external() → DataFrame
```

In [ ]:
try:
    from lic_dsf.load import load_instruments_from_workbook
except ImportError as exc:
    raise NotImplementedError(
        "Implement load_instruments_from_workbook(workbook_path, ...)"
    ) from exc

instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=False,
)
pd.DataFrame(
    [
        {
            "name": i.name,
            "grace": i.grace,
            "maturity": i.maturity,
            "interest": i.interest_rate,
            "discount": i.discount_rate,
            "disbursement_sum": sum(i.disbursements),
        }
        for i in instruments
    ]
)

## 3. Portfolio aggregate (target)

```python
portfolio = PresentValuePortfolio(instruments)
totals = portfolio.aggregate_external()  # DataFrame, same row index as external()
```

In [ ]:
try:
    from lic_dsf.pv import PresentValuePortfolio
except ImportError as exc:
    raise NotImplementedError(
        "Implement PresentValuePortfolio with aggregate_external() -> DataFrame"
    ) from exc

portfolio = PresentValuePortfolio(instruments)
totals = portfolio.aggregate_external()
assert isinstance(totals, pd.DataFrame)
totals.iloc[:, :10]

## Build order

1. ~~DataFrame `internal()` / `external()`~~ (done)
2. `load_instruments_from_workbook`
3. `PresentValuePortfolio.aggregate_external()`
4. Parity on **Eurobond** (nonzero disbursements), not MULTI1
5. IMF / cost-shock / ResFin / LC-NR later